# crewai.Knowledge




`Knowledge` is a collection of sources and setup for the vector store to save and query relevant context.
* `BaseKnowledgeSource`s (see [_KNOWN_SOURCES](https://github.com/crewAIInc/crewAI/blob/1.15.16/lib/crewai/src/crewai/knowledge/knowledge.py#L23-L31))
* `EmbedderConfig`
* `BaseKnowledgeStorage`

Key benefits of using [Knowledge](https://docs.crewai.com/v1.15.16/en/concepts/knowledge) in crewAI:
* Enhance agents with domain-specific information
* Support decisions with real-world data
* Maintain context across conversations
* Ground responses in factual information

`Knowledge` is a named entity (by `collection_name`).


`Knowledge` is a [Pydantic model](https://pydantic.dev/docs/validation/latest/concepts/models/).

<br>

```py
class Knowledge(BaseModel)
```


## class_method_map

[Among the tracked classes by MLflow](https://github.com/mlflow/mlflow/blob/v3.15.1/mlflow/crewai/__init__.py#L78).

<br>

```py
class_method_map.update({"crewai.Knowledge": ["query"]})
```

## SpanType.RETRIEVER

`crewai.Knowledge` is assigned [SpanType.RETRIEVER](https://github.com/mlflow/mlflow/blob/v3.15.1/mlflow/crewai/autolog.py#L242-L243) span type in MLflow.

## _get_span_type

[_get_span_type](https://github.com/mlflow/mlflow/blob/v3.15.1/mlflow/crewai/autolog.py#L201-L247) assigns span types to crewAI instances.

crewAI's Instance | Span Type
-|-
Agent | `AGENT`
BaseAgentExecutor | `MEMORY`
Crew | `CHAIN`
`EntityMemory` | `MEMORY`
Flow | `CHAIN`
Knowledge | `RETRIEVER`
LLM | `LLM`
`LongTermMemory` | `MEMORY`
`ShortTermMemory` | `MEMORY`
Task | `CHAIN`

# 🛠️ Set Up Environment

In [0]:
%pip install -qU "mlflow-skinny[databricks]>=3.15.1" "crewai[tools]>=1.15.16"
%restart_python

In [0]:
%pip show crewai

In [0]:
%pip show mlflow-skinny

# crewai/knowledge/knowledge.py


`crewai.Knowledge` is a class defined in `crewai/knowledge/knowledge.py`.

In [0]:
from crewai.knowledge.knowledge import Knowledge


`crewai.Knowledge` is re-exported in `crewai/__init__.py` for simplicity (and publicity = the official API).

In [0]:
from crewai import Knowledge

In [0]:
import mlflow

mlflow.crewai.autolog()


`Crew`s and `Agent`s can have dedicated `Knowledge`s.

* `Crew.knowledge_sources` - Knowledge sources for the crew.
* `BaseAgent.knowledge_sources` - Knowledge sources for the agent.

In [0]:
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource
from textwrap import dedent

# Create knowledge source with user preferences
content = "Users name is John. He is 30 years old and lives in San Francisco."
string_source = StringKnowledgeSource(
    content=content, 
    metadata={"preference": "personal"}
)
print(string_source)

In [0]:
import os

os.environ["OPENAI_API_KEY"] = "NA"

In [0]:
from crewai.rag.chromadb.config import ChromaDBConfig

config = ChromaDBConfig()
print(config)

In [0]:
# Get Databricks workspace URL from the notebook context
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
print(workspace_url)

In [0]:
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
print(token)

In [0]:
model_name = "databricks-meta-llama-3-3-70b-instruct"
model=f"databricks/{model_name}"

In [0]:
config.embedding_function.api_base = f"https://{workspace_url}/serving-endpoints"
config.embedding_function.api_key = token
config.embedding_function.model = model

In [0]:
from crewai.rag.chromadb.client import ChromaDBClient
import chromadb

chroma_client = chromadb.EphemeralClient(settings=config.settings, tenant=config.tenant, database=config.database)
client = ChromaDBClient(client=chroma_client, embedding_function=config.embedding_function)

In [0]:
import numpy as np
from crewai.rag.core.types import Documents, Embeddings
from crewai.rag.embeddings.providers.custom.embedding_callable import CustomEmbeddingFunction

class DatabricksEmbeddingFunction(CustomEmbeddingFunction):
    def __init__(self, client):
        self.client = client

    def __call__(self, input: Documents) -> Embeddings:
        return config.embedding_function(input)
    
    def name(self):
        return "databricks-embedding-function"

In [0]:
embedding_callable = DatabricksEmbeddingFunction(client=client)

In [0]:
embedding_callable(input=["hello", "world"])

In [0]:
from crewai.rag.embeddings.providers.custom.custom_provider import CustomProvider

embedder = CustomProvider(embedding_callable=DatabricksEmbeddingFunction)
print(embedder)

In [0]:
knowledge = Knowledge(collection_name="docs", sources=[string_source], embedder=embedder)

In [0]:
knowledge.query(["what do you know about John?"])